# AIKU 필적 분류 — Colab 파이프라인

`src/` 모듈을 그대로 호출만 한다 (로직 중복 작성 금지). 런타임은 GPU(T4 이상)로 설정할 것.

단계: ① 전처리(train patch 추출) → ② 임베딩 추출(train/test) → ③ 학습(CV) → ④ 추론(submission.csv)

⚠️ test 는 암호화되어 있다. `src/test_loader.py`, `src/embedding.py`, `src/inference.py` 의 test 경로는
복호화 이미지를 화면에 띄우거나 파일로 저장하지 않는다 — 이 노트북에서도 그런 셀을 추가하지 말 것.

## 0. Drive 마운트 & 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/writer_id_project'
# 최초 1회: 로컬에서 준비한 12-aikuthon/ 전체(train, test, src, sample_submission.csv 등)를
# PROJECT_DIR 에 업로드/동기화해 둔다. 이후 세션에서는 이 경로만 재사용.

import sys
sys.path.append(PROJECT_DIR)

## 1. 의존성 설치

In [ ]:
!pip install -q open_clip_torch opencv-python-headless scikit-learn scikit-image joblib lightgbm pytesseract
!apt-get -q install -y tesseract-ocr  # features.py(OCR 메타정보)용 — 이번 MVP 단계에서는 미사용

## 2. 재현성 시드 고정

In [ ]:
from src import config
config.set_seed()
print('WRITERS =', config.WRITERS)

## ① 전처리 — train PNG에서 손글씨 patch 추출

In [ ]:
from src.preprocessing import run as run_preprocessing
run_preprocessing(config.TRAIN_DIR, config.CACHE_DIR)

## ② 임베딩 추출 (train / test, backbone 선택)

In [ ]:
BACKBONE = 'dinov2'  # 또는 'clip'
DEVICE = 'cuda'

from src.embedding import embed_train, embed_test
embed_train(config.CACHE_DIR, BACKBONE, DEVICE)
embed_test(config.TEST_ROOT, config.SUBMISSION_TEMPLATE, BACKBONE, DEVICE)

## ③ 학습 — StratifiedGroupKFold CV + class-weighted LogisticRegression

In [ ]:
from src.train import load_train_embeddings, cross_validate, fit_final

vectors, labels, page_ids = load_train_embeddings(BACKBONE)
best_c = cross_validate(vectors, labels, page_ids, n_splits=5, c_grid=[0.01, 0.1, 1.0, 10.0])
fit_final(vectors, labels, best_c, BACKBONE)

## ④ 추론 — submission.csv 생성

In [ ]:
from src.inference import generate_submission
generate_submission(BACKBONE, DEVICE, config.SUBMISSION_OUT)